In [2]:
import pandas as pd 
import json 
from tqdm.auto import tqdm
from openai import OpenAI

In [3]:


client = OpenAI()

In [4]:
df = pd.read_csv('../data/cleaned_data.csv')

In [5]:

documents = df.to_dict(orient='records')

In [6]:
print(documents[0])

{'id': 1, 'task': 'Write a 2-page project summary', 'category': 'work', 'difficulty': 'medium', 'duration_estimate': 45, 'framework_name': 'Time Blocking', 'reasoning': 'Time Blocking helps allocate a clear writing window.', 'instructions': 'Block 45 minutes, outline key points, write summary, revise.', 'tags': 'work;writing;planning'}


## Evaluation

In [7]:
prompt_template = """
You emulate a user of our Productivity Advisor application.
Formulate 5 questions this user might ask based on a provided productivity task.
Make the questions specific to this task.
The questions should be complete, practical, and not too short.
Use as few exact words as possible from the record.

The record:

task: {task}
category: {category}
difficulty: {difficulty}
duration_estimate: {duration_estimate}
instructions: {instructions}
reasoning: {reasoning}
tags: {tags}

Provide the output in parsable JSON without using code blocks:

{{"questions": ["question1", "question2", ..., "question5"]}}

""".strip()

In [8]:
prompt = prompt_template.format(**documents[0])

print(prompt)

You emulate a user of our Productivity Advisor application.
Formulate 5 questions this user might ask based on a provided productivity task.
Make the questions specific to this task.
The questions should be complete, practical, and not too short.
Use as few exact words as possible from the record.

The record:

task: Write a 2-page project summary
category: work
difficulty: medium
duration_estimate: 45
instructions: Block 45 minutes, outline key points, write summary, revise.
reasoning: Time Blocking helps allocate a clear writing window.
tags: work;writing;planning

Provide the output in parsable JSON without using code blocks:

{"questions": ["question1", "question2", ..., "question5"]}


In [9]:
model='gpt-4o-mini'
def llm(prompt, model=model):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [10]:
questions = llm(prompt)
print(questions)

{"questions": ["What are the most important points I should include in my outline for the project summary?", "How can I effectively use the 45 minutes I have blocked to ensure I complete the writing?", "What strategies can I employ to revise my summary effectively after writing it?", "Are there any templates or examples of project summaries that could guide my writing?", "How can I minimize distractions during my dedicated writing time for this task?"]}


In [11]:
import json

print(json.loads(questions))

{'questions': ['What are the most important points I should include in my outline for the project summary?', 'How can I effectively use the 45 minutes I have blocked to ensure I complete the writing?', 'What strategies can I employ to revise my summary effectively after writing it?', 'Are there any templates or examples of project summaries that could guide my writing?', 'How can I minimize distractions during my dedicated writing time for this task?']}


In [12]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [13]:

results = {}

In [14]:
for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions_raw = generate_questions(doc)
    questions = json.loads(questions_raw)
    results[doc_id] = questions['questions']

  0%|          | 0/250 [00:00<?, ?it/s]

In [15]:
final_results = []

for doc_id, questions in results.items():
    for q in questions:
        final_results.append((doc_id, q))

In [16]:
final_results[0]

(1, 'What key points should I include in my outline for the project summary?')

In [17]:
df_results = pd.DataFrame(final_results, columns=['id', 'question'])

In [18]:
df_results.to_csv('../data/ground-truth-retrieval.csv', index=False)

In [19]:
!head ../data/ground-truth-retrieval.csv

id,question
1,What key points should I include in my outline for the project summary?
1,How can I effectively manage my time during the 45-minute writing session?
1,What strategies can I use to ensure my summary is concise and clear within the 2-page limit?
1,How should I approach the revision process after completing the initial draft?
1,Are there any specific writing techniques that can help improve the quality of my project summary?
2,What is the best way to categorize different food items while organizing the pantry shelves?
2,How can I efficiently clean each shelf after removing all items from it?
2,Should I use any specific organizing tools or containers to help with the restocking process?
2,What strategies can I use to maintain the order in my pantry once I finish organizing it?
